# 109 — PII Redaction Pipeline
## Bi-directional PII sanitization with Microsoft Presidio
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/109-pii-redaction-pipeline/pii_redaction_pipeline_workbook.ipynb)

LLMs memorise and recombine everything in their context window. Feed a model a document containing a name, Social Security Number, or credit card number, and there is a real chance it will quote that data verbatim in its response — even if you never asked. This is not a theoretical risk: GDPR Article 25 (data-protection-by-design) and HIPAA's technical safeguards both require that PII be protected **before** it reaches any processing system, including AI inference.

This workshop teaches the **dual-stage redaction pattern** using **Microsoft Presidio** — an open-source NLP-based anonymization engine. The two stages are:

- **Stage 1 — Pre-ingestion**: Scrub PII from every document _before_ the LLM sees it. The model never touches the raw data.
- **Stage 2 — Post-generation**: Scrub any PII the LLM still produces in its reply _before_ it is delivered to the user.

Defense-in-depth: even if one stage misses something, the other catches it.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — Why PII leaks happen and the dual-stage model |
| 2 | **Setup** — Install Presidio, spaCy, and environment |
| 3 | **Analyzer** — How Presidio detects PII entities |
| 4 | **Anonymizer** — Replacing PII with safe placeholders |
| 5 | **The `redact()` tool** — Wrapping detection + anonymization |
| 6 | **Stage 1** — Pre-ingestion redaction before LLM call |
| 7 | **Stage 2** — Post-generation redaction of LLM output |
| 8 | **Full pipeline** — `run_pipeline()` end-to-end |
| 9 | **Multi-document run** — Processing the sample corpus |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab (free tier works fine)
- `OPENAI_API_KEY` in `.env` file or Colab Secrets
- Internet access for `en_core_web_lg` spaCy model download

### Key References
> Lison, P. et al. (2021). *Anonymisation Models for Clinical NLP*. ACL Anthology.
> Microsoft Presidio — [https://microsoft.github.io/presidio/](https://microsoft.github.io/presidio/)
> spaCy Named Entity Recognition — [https://spacy.io/usage/linguistic-features#named-entities](https://spacy.io/usage/linguistic-features#named-entities)
> GDPR Article 25 — Data Protection by Design and by Default
> HIPAA Technical Safeguards — 45 CFR § 164.312

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "presidio-analyzer", "presidio-anonymizer",
         "spacy", "langchain-openai", "langchain-core", "python-dotenv"],
        check=True
    )
    # Download the large English spaCy model required by Presidio
    subprocess.run(
        [sys.executable, "-m", "spacy", "download", "en_core_web_lg"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")
    print("Make sure you have run: pip install presidio-analyzer presidio-anonymizer spacy langchain-openai")
    print("And: python -m spacy download en_core_web_lg")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded API key from Colab Secrets.")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded environment from .env file.")

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — Concepts: Why PII Leaks Happen

### The fundamental problem

A language model is a statistical system trained to continue text. When you inject a document into its context, it treats every token — including PII — as ground truth to draw upon. Ask it to summarise the document and it may faithfully reproduce names, addresses, and account numbers in its reply.

### Three attack surfaces

```
┌─────────────────────────────────────────────────────────────────┐
│  RAW DOCUMENT                                                   │
│  "John Smith (SSN: 123-45-6789) owes $450 on card 4532-xxxx"  │
└──────────────────────┬──────────────────────────────────────────┘
                       │  ← Attack surface 1: unredacted input
                       ▼
               ┌───────────────┐
               │   LLM CONTEXT │  ← Attack surface 2: model memorisation
               │   (gpt-4o-mini│
               └───────┬───────┘
                       │
                       ▼
  ┌──────────────────────────────────────────────────────────────┐
  │  LLM RESPONSE                                               │
  │  "The customer John Smith's SSN is 123-45-6789 ..."         │
  └────────────────────┬─────────────────────────────────────────┘
                       │  ← Attack surface 3: unredacted output
                       ▼
                    USER
```

### The dual-stage defense

```
┌─────────────────────────────────────────────────────────────────┐
│  RAW DOCUMENT (with PII)                                       │
└──────────────────────┬──────────────────────────────────────────┘
                       │
              ┌────────▼────────┐
              │  STAGE 1        │  Presidio detects + anonymizes
              │  Pre-ingestion  │  Names → <PERSON>
              │  Redaction      │  SSNs  → <US_SSN>
              └────────┬────────┘  Cards  → <CREDIT_CARD>
                       │
               ┌───────▼───────┐
               │  LLM CONTEXT  │  Model NEVER sees raw PII
               │  (clean doc)  │
               └───────┬───────┘
                       │
              ┌────────▼────────┐
              │  STAGE 2        │  Catch any leakage in output
              │  Post-generation│
              │  Redaction      │
              └────────┬────────┘
                       │
                    USER (safe)
```

### Entity types Presidio detects (default recognizers)

| Category | Entities |
|----------|----------|
| Identity | `PERSON`, `EMAIL_ADDRESS`, `PHONE_NUMBER` |
| Financial | `CREDIT_CARD`, `IBAN_CODE`, `US_BANK_NUMBER` |
| Government | `US_SSN`, `US_DRIVER_LICENSE`, `US_PASSPORT` |
| Network | `IP_ADDRESS`, `URL`, `DOMAIN_NAME` |
| Location | `LOCATION` |

Presidio uses both **rule-based recognizers** (regex + checksums for structured data like credit cards) and **NLP-based recognizers** (spaCy NER for names and locations). This hybrid approach gives high precision for structured PII and reasonable recall for free-text PII.

---
## Part 2 — Setup: Sample Data

The sample documents intentionally contain multiple types of PII so we can see exactly what Presidio finds. These are representative of real-world CRM notes, support tickets, and meeting logs.

In [ ]:
# Sample documents from src/tools.py — realistic customer-facing text with PII
SAMPLE_DOCUMENTS = [
    """Customer complaint from John Smith (john.smith@gmail.com, 555-867-5309).
    His SSN is 123-45-6789 and credit card 4532-1234-5678-9012.
    He lives at 742 Evergreen Terrace, Springfield, IL 62701.""",

    """Meeting notes: Sarah Connor (sarah.c@skynet.ai) called from +1-800-555-0199.
    Her employee ID is EMP-9841 and IP address 192.168.1.42 flagged suspicious.""",
]

# A simulated LLM response that still leaks PII (what Stage 2 catches)
SAMPLE_LLM_RESPONSE = """Based on our records, the customer John Doe (DOB: 1985-03-15) with
account number ACC-789456 and phone 555-234-5678 has been escalated.
Their email john.doe@example.com should receive the refund confirmation."""

print("Document 1:")
print(SAMPLE_DOCUMENTS[0])
print()
print("Document 2:")
print(SAMPLE_DOCUMENTS[1])
print()
print("Sample LLM response (contains PII that Stage 2 must catch):")
print(SAMPLE_LLM_RESPONSE)

---
## Part 3 — The Presidio AnalyzerEngine

The `AnalyzerEngine` is the detection half of Presidio. It scans text and returns a list of `RecognizerResult` objects — each describing an entity type, its position in the text (`start`, `end`), and a confidence `score` (0.0 to 1.0).

### How the analyzer works internally

```
Input text
    │
    ├──► RegexRecognizer     (EMAIL_ADDRESS, CREDIT_CARD, US_SSN, PHONE_NUMBER)
    │    Uses validated patterns + optional checksums (Luhn for credit cards)
    │
    ├──► SpacyRecognizer     (PERSON, LOCATION, ORG)
    │    Runs en_core_web_lg NER model
    │
    └──► Results merged, deduplicated, scored
```

The returned `score` is meaningful:
- **1.0** — structural certainty (e.g., Luhn-valid credit card)
- **0.85** — high-confidence NER
- **0.5** — context-dependent, could be a false positive

By default, Presidio uses a score threshold of **0.35** before reporting an entity.

In [ ]:
from presidio_analyzer import AnalyzerEngine

# Build the analyzer (loads spaCy model on first call — takes ~5 seconds)
analyzer = AnalyzerEngine()

# Analyze Document 1
results = analyzer.analyze(text=SAMPLE_DOCUMENTS[0], language="en")

print(f"Found {len(results)} PII entities in Document 1:\n")
print(f"{'Entity Type':<20} {'Score':>6}  {'Text':<35}  Start-End")
print("-" * 75)
for r in sorted(results, key=lambda x: x.start):
    span = SAMPLE_DOCUMENTS[0][r.start:r.end]
    print(f"{r.entity_type:<20} {r.score:>6.2f}  {repr(span):<35}  {r.start}-{r.end}")

In [ ]:
# Analyze Document 2
results2 = analyzer.analyze(text=SAMPLE_DOCUMENTS[1], language="en")

print(f"Found {len(results2)} PII entities in Document 2:\n")
print(f"{'Entity Type':<20} {'Score':>6}  {'Text':<35}  Start-End")
print("-" * 75)
for r in sorted(results2, key=lambda x: x.start):
    span = SAMPLE_DOCUMENTS[1][r.start:r.end]
    print(f"{r.entity_type:<20} {r.score:>6.2f}  {repr(span):<35}  {r.start}-{r.end}")

---
## Part 4 — The Presidio AnonymizerEngine

Once entities are detected, the `AnonymizerEngine` replaces them. The default operator is **replace** — it substitutes each detected span with a placeholder like `<PERSON>` or `<US_SSN>`.

### Anonymization strategies

| Strategy | Example output | Use case |
|----------|---------------|----------|
| **Replace** (default) | `<PERSON>` | Safe for display, loses context |
| **Redact** | `` (empty) | Maximum safety |
| **Hash** | `sha256:a1b2c3...` | Consistent pseudonym for linking |
| **Mask** | `****-****-5678` | Partial visibility (last 4 of phone) |
| **Encrypt** | `AES:xyz...` | Reversible (for authorized retrieval) |
| **Custom** | `JOHN → Customer_1` | Consistent fake names |

For an LLM pipeline, **replace** is the right default — it tells the model *something was here* without revealing what, preserving sentence structure so the model can still answer questions about the document's topic.

In [ ]:
from presidio_anonymizer import AnonymizerEngine

anonymizer = AnonymizerEngine()

# Anonymize using the results we already computed
anonymized = anonymizer.anonymize(
    text=SAMPLE_DOCUMENTS[0],
    analyzer_results=results   # from the previous cell
)

print("Original:")
print(SAMPLE_DOCUMENTS[0])
print()
print("Anonymized:")
print(anonymized.text)
print()
print("Items replaced:", len(anonymized.items))
for item in anonymized.items:
    print(f"  {item.entity_type}: start={item.start}, end={item.end}, operator={item.operator}")

---
## Part 5 — The `redact()` Tool

The `redact()` function from `src/tools.py` wraps detection and anonymization into a single call with a clean interface. It also returns structured metadata about what was found — useful for audit logging (which entity types were detected, how many, with what confidence).

```
redact(text, analyzer, anonymizer)
    │
    ├── analyzer.analyze(text) → list[RecognizerResult]
    ├── anonymizer.anonymize(text, results) → AnonymizerResult
    └── returns (clean_text: str, entities: list[dict])
               where entities = [{type, score, text}, ...]
```

The entity metadata (type, score, original text) can be written to an audit log for compliance purposes — you can prove which PII was found and redacted without storing the PII itself.

In [ ]:
# The redact() function from src/tools.py — copied exactly
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine


def build_engines() -> tuple[AnalyzerEngine, AnonymizerEngine]:
    """Instantiate and return both Presidio engines."""
    analyzer = AnalyzerEngine()
    anonymizer = AnonymizerEngine()
    return analyzer, anonymizer


def redact(text: str, analyzer: AnalyzerEngine, anonymizer: AnonymizerEngine) -> tuple[str, list]:
    """Detect PII in text and return anonymized version plus entity metadata."""
    results = analyzer.analyze(text=text, language="en")
    redacted = anonymizer.anonymize(text=text, analyzer_results=results)
    entities_found = [
        {"type": r.entity_type, "score": round(r.score, 2), "text": text[r.start:r.end]}
        for r in results
    ]
    return redacted.text, entities_found


# Demonstrate redact() on Document 2
analyzer_engine, anonymizer_engine = build_engines()
clean_text, entities = redact(SAMPLE_DOCUMENTS[1], analyzer_engine, anonymizer_engine)

print("Input:")
print(SAMPLE_DOCUMENTS[1])
print()
print("Output (clean_text):")
print(clean_text)
print()
print("Entities found (for audit log):")
for e in entities:
    print(f"  type={e['type']}, score={e['score']}, original_text={repr(e['text'])}")

---
## Part 6 — Stage 1: Pre-Ingestion Redaction

Stage 1 runs **before** any LLM call. The goal is to ensure the model context never contains raw PII.

### The grounded system prompt

The system prompt from `src/workflow.py` reinforces this boundary:

```
REDACTION_SYSTEM = "You are a helpful assistant. Answer based only on the provided document."
```

"Based only on the provided document" does two things:
1. It prevents the model from speculating beyond the context (reducing hallucination)
2. Since the document is already clean, it grounds the model in placeholder text only

### Document wrapper structure

```
REDACTION_USER = "Document:\n{document}\n\nQuestion: {question}"
```

Separating document from question reduces prompt injection risk — a malicious document cannot easily override the question because they are structurally distinct parts of the message.

In [ ]:
# Stage 1: Pre-ingestion redaction
raw_doc = SAMPLE_DOCUMENTS[0]

# Build fresh engines (in production, reuse these across requests for performance)
analyzer_engine, anonymizer_engine = build_engines()

# Redact before LLM ever sees it
clean_doc, pre_entities = redact(raw_doc, analyzer_engine, anonymizer_engine)

print("=" * 60)
print("RAW DOCUMENT (what user submitted, contains PII):")
print("=" * 60)
print(raw_doc)
print()
print("=" * 60)
print("CLEAN DOCUMENT (what the LLM will see, PII scrubbed):")
print("=" * 60)
print(clean_doc)
print()
print(f"Entities redacted from input ({len(pre_entities)} total):")
for e in pre_entities:
    print(f"  [{e['type']}] score={e['score']}")

---
## Part 7 — LLM Call on Clean Context

With the clean document in hand, we call the LLM. Notice that the model:
- Receives only `<PERSON>`, `<EMAIL_ADDRESS>`, etc. — not the real values
- Can still answer questions about structure, sentiment, and topic
- May attempt to refer to the placeholder tokens in its response

That last point is why Stage 2 still matters — the model might write "the customer <PERSON> reported..." and if the Stage 2 redactor is not run, a future pipeline step might interpret `<PERSON>` tags incorrectly.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Prompts from src/workflow.py
REDACTION_SYSTEM = "You are a helpful assistant. Answer based only on the provided document."
REDACTION_USER = "Document:\n{document}\n\nQuestion: {question}"

QUESTION = "Summarize the key details about the person mentioned."

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = [
    SystemMessage(content=REDACTION_SYSTEM),
    HumanMessage(content=REDACTION_USER.format(document=clean_doc, question=QUESTION)),
]

raw_response = llm.invoke(messages).content

print("LLM response (before Stage 2 redaction):")
print("-" * 60)
print(raw_response)

---
## Part 8 — Stage 2: Post-Generation Redaction

Stage 2 runs **after** the LLM responds, before the response is delivered to the user. This catches:

1. **Hallucinated PII** — the model invented a realistic-looking name or phone number
2. **Context bleed** — the model somehow recombined training data with the current context
3. **Placeholder leakage** — the model passed through a `<PERSON>` tag that a downstream system might mishandle
4. **Injected PII** — a malicious document included PII designed to survive prompt injection

Stage 2 uses exactly the same `redact()` function — the same Presidio engines, the same entity types. This consistency means both stages stay in sync as you extend the entity list.

In [ ]:
# Stage 2: Post-generation redaction of the LLM's output
clean_response, post_entities = redact(raw_response, analyzer_engine, anonymizer_engine)

print("RAW LLM RESPONSE (may contain PII):")
print("-" * 60)
print(raw_response)
print()
print("FINAL RESPONSE (delivered to user, PII scrubbed):")
print("-" * 60)
print(clean_response)
print()
if post_entities:
    print(f"Stage 2 caught {len(post_entities)} additional entities in the LLM output:")
    for e in post_entities:
        print(f"  [{e['type']}] score={e['score']}, text={repr(e['text'])}")
else:
    print("Stage 2: no additional PII detected in LLM output (Stage 1 was sufficient).")

---
## Part 9 — The Full `run_pipeline()` Function

The `run_pipeline()` function from `src/workflow.py` composes both stages into a single call and returns a structured result dictionary for inspection, logging, and testing.

### Return value schema

```python
{
    "original_doc":     str,   # raw input — NEVER log this in production
    "pre_redacted_doc": str,   # clean doc sent to LLM
    "pre_entities":     list,  # PII found in input (type, score only — not text)
    "raw_llm_response": str,   # LLM output before Stage 2
    "final_response":   str,   # clean output delivered to user
    "post_entities":    list,  # any PII Stage 2 caught in LLM output
}
```

In production, `original_doc` and `raw_llm_response` should not be persisted — only the clean versions and entity-type metadata (no original text values) should go into audit logs.

In [ ]:
# The full run_pipeline() function from src/workflow.py — copied exactly

def run_pipeline(raw_document: str, question: str) -> dict:
    """
    Two-stage PII pipeline:
    1. Pre-ingestion: redact PII from document before sending to LLM
    2. Post-generation: redact any PII the LLM leaks in its response
    """
    analyzer, anonymizer = build_engines()

    # Stage 1: scrub input before LLM sees it
    clean_doc, pre_entities = redact(raw_document, analyzer, anonymizer)

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    messages = [
        SystemMessage(content=REDACTION_SYSTEM),
        HumanMessage(content=REDACTION_USER.format(document=clean_doc, question=question)),
    ]
    raw_response = llm.invoke(messages).content

    # Stage 2: scrub any PII the LLM included in its response
    clean_response, post_entities = redact(raw_response, analyzer, anonymizer)

    return {
        "original_doc": raw_document,
        "pre_redacted_doc": clean_doc,
        "pre_entities": pre_entities,
        "raw_llm_response": raw_response,
        "final_response": clean_response,
        "post_entities": post_entities,
    }


print("run_pipeline() defined — ready to process documents.")

---
## Part 10 — Multi-Document Processing

The `main.py` entry point loops over all sample documents, calls `run_pipeline()`, and prints a summary. This is the production pattern: process each document independently, log entity-type counts, deliver only the final clean response.

The `show_entities()` helper formats the entity list concisely — type + score — without printing the original PII text. This is intentional: even in debug output, you want to avoid logging PII.

In [ ]:
# Helper from main.py — logs entity summary without printing PII text
def show_entities(label: str, entities: list[dict]) -> None:
    if entities:
        types = ", ".join(f"{e['type']}({e['score']})" for e in entities[:5])
        print(f"  {label}: [{types}{'...' if len(entities) > 5 else ''}]")
    else:
        print(f"  {label}: none detected")


# Run the full pipeline over all sample documents
print("PII Redaction Pipeline — Microsoft Presidio bi-directional sanitization")
print("=" * 70)
print("Stage 1: redact document before LLM ingestion")
print("Stage 2: redact LLM response before delivery\n")

results_all = []
for i, doc in enumerate(SAMPLE_DOCUMENTS, 1):
    print(f"--- Document {i} ---")
    result = run_pipeline(doc, QUESTION)
    results_all.append(result)

    show_entities("Pre-ingestion entities found", result["pre_entities"])
    show_entities("Post-generation entities found", result["post_entities"])

    print(f"\n  Pre-redacted doc (first 120 chars):")
    print(f"    {result['pre_redacted_doc'][:120].replace(chr(10), ' ')}...")
    print(f"\n  Final response (delivered to user):")
    print(f"    {result['final_response'][:200].replace(chr(10), ' ')}")
    print()

---
## Part 11 — Inspecting Stage 2 on Synthetic LLM Leakage

To clearly demonstrate Stage 2's purpose, let's run `redact()` directly on the `SAMPLE_LLM_RESPONSE` — a response that was crafted to look like a model that leaked PII despite seeing a clean context (hallucination or training data bleed).

In [ ]:
# Demonstrate Stage 2 catching hallucinated / leaked PII in an LLM response
analyzer_engine2, anonymizer_engine2 = build_engines()

clean_llm, llm_entities = redact(SAMPLE_LLM_RESPONSE, analyzer_engine2, anonymizer_engine2)

print("Synthetic LLM response (simulating PII leakage):")
print("-" * 60)
print(SAMPLE_LLM_RESPONSE)
print()
print("After Stage 2 redaction:")
print("-" * 60)
print(clean_llm)
print()
print(f"Stage 2 caught {len(llm_entities)} entities:")
for e in llm_entities:
    print(f"  [{e['type']}] score={e['score']}, original={repr(e['text'])}")

---
## Part 12 — Understanding Score Thresholds and False Positives

Presidio's default score threshold is 0.35 — anything above this is reported as a detected entity. This means:

- **Higher threshold** → fewer false positives, but risk of missing real PII (higher false negative rate)
- **Lower threshold** → catches more PII, but over-redacts legitimate text

For HIPAA/GDPR-compliant systems, you generally want **aggressive** redaction (lower threshold) because a missed SSN is far more damaging than an over-redacted sentence.

You can control the threshold per call and even per entity type.

In [ ]:
# Demonstrate threshold control — analyze at different score thresholds
test_text = """Call Dr. Maria Santos at (650) 555-0123 or maria@hospital.org.
Patient ID: 98765. Insurance: BCBS. Next appointment: 2024-03-15."""

analyzer3 = AnalyzerEngine()

for threshold in [0.1, 0.35, 0.6, 0.85]:
    results_t = analyzer3.analyze(text=test_text, language="en", score_threshold=threshold)
    print(f"Threshold {threshold:.2f} → {len(results_t)} entities: ", end="")
    print(", ".join(f"{r.entity_type}({r.score:.2f})" for r in sorted(results_t, key=lambda x: x.start)))

---
## Part 13 — Production Considerations

### Performance

Loading `en_core_web_lg` takes ~2-5 seconds and ~700MB RAM. In production:
- **Singleton pattern**: build engines once at application startup, share across requests
- **The current `build_engines()` call inside `run_pipeline()` is for tutorial clarity** — in production, pass pre-built engines in as arguments

### Custom recognizers

Presidio supports adding your own patterns:

```python
from presidio_analyzer import PatternRecognizer, Pattern

mrn_recognizer = PatternRecognizer(
    supported_entity="MEDICAL_RECORD_NUMBER",
    patterns=[Pattern(name="MRN", regex=r"MRN-\d{7}", score=0.9)]
)
analyzer.registry.add_recognizer(mrn_recognizer)
```

### What the pipeline does NOT cover

| Gap | Mitigation |
|-----|------------|
| PII in non-English text | Add `language="fr"` / multilingual models |
| PII in images / PDFs | OCR → Presidio (Tesseract + Presidio Image Redactor) |
| Novel PII patterns not in training | Custom recognizers + deny-list |
| PII in structured JSON/CSV fields | Field-level redaction before text conversion |
| De-anonymization attacks | k-anonymity, differential privacy for aggregate data |

In [ ]:
# Demonstrate singleton pattern — build once, reuse many times
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
import time

# Build once (this is the slow part)
t0 = time.time()
shared_analyzer = AnalyzerEngine()
shared_anonymizer = AnonymizerEngine()
build_time = time.time() - t0
print(f"Engine build time: {build_time:.2f}s (done once at startup)")

# Subsequent calls are fast
times = []
for doc in SAMPLE_DOCUMENTS:
    t0 = time.time()
    _ = redact(doc, shared_analyzer, shared_anonymizer)
    times.append(time.time() - t0)

print(f"Per-document redaction time (pre-built engines):")
for i, t in enumerate(times, 1):
    print(f"  Document {i}: {t*1000:.1f}ms")

---
## Part 14 — Pipeline Result Inspection

Let's inspect the full structured output from one `run_pipeline()` call — showing every key in the return dictionary and what it contains.

In [ ]:
# Inspect a full result dictionary
result = results_all[0]  # Document 1 result from the multi-document run above

print("Full pipeline result for Document 1:")
print()

print("KEY: original_doc")
print(f"  (raw input — would NOT be logged in production)")
print(f"  {result['original_doc'][:100].replace(chr(10), ' ')}...")
print()

print("KEY: pre_redacted_doc")
print(f"  (what the LLM saw)")
print(f"  {result['pre_redacted_doc'][:100].replace(chr(10), ' ')}...")
print()

print("KEY: pre_entities")
print(f"  {len(result['pre_entities'])} entities detected in input:")
for e in result['pre_entities']:
    print(f"    {e['type']} (score={e['score']})")
print()

print("KEY: raw_llm_response")
print(f"  (before Stage 2 — would NOT be logged in production)")
print(f"  {result['raw_llm_response'][:150].replace(chr(10), ' ')}...")
print()

print("KEY: final_response")
print(f"  (what the user receives)")
print(f"  {result['final_response'][:150].replace(chr(10), ' ')}")
print()

print("KEY: post_entities")
print(f"  {len(result['post_entities'])} entities caught by Stage 2:")
if result['post_entities']:
    for e in result['post_entities']:
        print(f"    {e['type']} (score={e['score']})")
else:
    print("  (none — Stage 1 was sufficient)")

---
## Exercises

### Exercise 1 — Add a custom entity recognizer

The sample documents contain `EMP-9841` (an employee ID). Presidio does not detect this by default. Add a custom `PatternRecognizer` for employee IDs matching the pattern `EMP-\d{4}` and verify it is detected.

**Hint**: Use `presidio_analyzer.PatternRecognizer` and `presidio_analyzer.Pattern`, then add it to `analyzer.registry`.

---

### Exercise 2 — Mask instead of replace

Modify the anonymization so that `PHONE_NUMBER` entities are **masked** (show last 4 digits) rather than replaced with `<PHONE_NUMBER>`. The `AnonymizerEngine` accepts an `operators` parameter — a dict mapping entity type to an `OperatorConfig`.

**Hint**: Use `from presidio_anonymizer.entities import OperatorConfig` and configure `"mask"` as the operator with `masking_char="*"` and `chars_to_mask=7`.

---

### Exercise 3 — Audit log function

Write a function `build_audit_entry(result: dict) -> dict` that takes a `run_pipeline()` result and returns a GDPR-safe audit entry. The audit entry must:
- Include counts of entity types found at each stage (not the original text)
- Include the final clean response
- NOT include `original_doc` or `raw_llm_response`
- Include a timestamp

In [ ]:
# ── Answer Key — Exercise 1: Custom entity recognizer for employee IDs ──────

from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine

# Build base engines
ex1_analyzer = AnalyzerEngine()
ex1_anonymizer = AnonymizerEngine()

# Define custom recognizer for employee IDs like EMP-9841
emp_recognizer = PatternRecognizer(
    supported_entity="EMPLOYEE_ID",
    patterns=[
        Pattern(
            name="employee_id_pattern",
            regex=r"EMP-\d{4}",
            score=0.9
        )
    ]
)

# Register it with the analyzer
ex1_analyzer.registry.add_recognizer(emp_recognizer)

# Test on Document 2 which contains EMP-9841
doc2_results = ex1_analyzer.analyze(text=SAMPLE_DOCUMENTS[1], language="en")

print("Exercise 1 — Entities detected in Document 2 (with custom EMPLOYEE_ID recognizer):")
print()
for r in sorted(doc2_results, key=lambda x: x.start):
    span = SAMPLE_DOCUMENTS[1][r.start:r.end]
    print(f"  {r.entity_type:<22} score={r.score:.2f}  text={repr(span)}")

# Verify EMP-9841 is found
emp_found = any(r.entity_type == "EMPLOYEE_ID" for r in doc2_results)
print(f"\nEMPLOYEE_ID detected: {emp_found}")

# Anonymize with custom recognizer
anonymized_ex1 = ex1_anonymizer.anonymize(
    text=SAMPLE_DOCUMENTS[1],
    analyzer_results=doc2_results
)
print(f"\nAnonymized text:\n{anonymized_ex1.text}")

In [ ]:
# ── Answer Key — Exercise 2: Mask phone numbers instead of replacing ─────────

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

ex2_analyzer = AnalyzerEngine()
ex2_anonymizer = AnonymizerEngine()

# Analyze Document 1 to find phone numbers
ex2_results = ex2_analyzer.analyze(text=SAMPLE_DOCUMENTS[0], language="en")

# Configure operators: mask PHONE_NUMBER, use default replace for everything else
operators = {
    "PHONE_NUMBER": OperatorConfig(
        operator_name="mask",
        params={
            "masking_char": "*",
            "chars_to_mask": 7,
            "from_end": False,
        }
    ),
    "DEFAULT": OperatorConfig(operator_name="replace", params={}),
}

anonymized_ex2 = ex2_anonymizer.anonymize(
    text=SAMPLE_DOCUMENTS[0],
    analyzer_results=ex2_results,
    operators=operators
)

print("Exercise 2 — Masked phone, replaced everything else:")
print()
print("Original:")
print(SAMPLE_DOCUMENTS[0])
print()
print("Anonymized (phone masked, others replaced):")
print(anonymized_ex2.text)

In [ ]:
# ── Answer Key — Exercise 3: GDPR-safe audit log function ────────────────────

from datetime import datetime, timezone
from collections import Counter
import json


def build_audit_entry(result: dict) -> dict:
    """
    Build a GDPR-safe audit entry from a run_pipeline() result.

    Includes:
    - Entity type counts at each stage (not original text)
    - Final clean response
    - Timestamp

    Excludes:
    - original_doc (contains raw PII)
    - raw_llm_response (may contain PII)
    """
    pre_counts = Counter(e["type"] for e in result["pre_entities"])
    post_counts = Counter(e["type"] for e in result["post_entities"])

    return {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "stage1_entity_counts": dict(pre_counts),
        "stage2_entity_counts": dict(post_counts),
        "total_entities_redacted": len(result["pre_entities"]) + len(result["post_entities"]),
        "final_response": result["final_response"],
        # Explicitly exclude PII-bearing fields
        # original_doc: NOT included
        # raw_llm_response: NOT included
    }


# Test on Document 1 result
audit = build_audit_entry(results_all[0])

print("Exercise 3 — GDPR-safe audit entry:")
print(json.dumps(audit, indent=2))
print()
print("Verification — no PII fields present:")
print(f"  'original_doc' in audit: {'original_doc' in audit}")
print(f"  'raw_llm_response' in audit: {'raw_llm_response' in audit}")
print(f"  'final_response' in audit: {'final_response' in audit}  (safe — already redacted)")

---
## Workshop Complete

You have built a production-grade dual-stage PII redaction pipeline:

- **Stage 1** prevents PII from entering LLM context — the model never sees it
- **Stage 2** catches any PII the model produces in its output — the user never receives it
- **`redact()`** combines Presidio's detection and anonymization in a single auditable call
- **Custom recognizers** extend the system to domain-specific entity types
- **Operator configs** let you choose masking strategy per entity type
- **Audit entries** provide GDPR/HIPAA compliance evidence without storing PII

### Key takeaways

1. Never trust the LLM to self-censor — enforce redaction at the infrastructure level
2. Defense-in-depth: two independent stages beat one perfect one
3. Build engines once, reuse them — `AnalyzerEngine` initialization is expensive
4. Log entity types, not entity values — the count proves compliance; the value creates liability

---

**Next: Example 110 — [Structured Output Validation](../110-structured-output-validation/)**

Enforce strict JSON schemas on LLM output with Pydantic and LangChain's `.with_structured_output()` — so your pipeline never fails on a malformed response.